# 06. PDF 파서 3종 + 기본 OCR — 표는 어디서 깨지나

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/acho98/gunyang-ai-api/blob/main/labs/06_pdf_parsers_ocr.ipynb)

> 🧭 **스토리** — 규정집 PDF를 AI에게 읽히려면 먼저 **글자와 표를 꺼내야** 한다. 인사팀이 준 파일은 두 가지다. 워드에서 저장한 **디지털 PDF**와, 종이를 스캔한 **이미지**. 같은 문서인데 꺼내는 방법도, 깨지는 곳도 다르다.
>
> 📖 **이론 연결** — 이론2 p5~24 (PDF 구조·파서 비교·표 추출·OCR 원리·OCR의 한계) · ⏱ **14분** · 🖥 **GPU 권장**
>
> GPU(T4)가 있으면 빠릅니다. 없어도 CPU로 돌아갑니다(느림).
> 💡 **앞 노트북을 열어 둔 상태라면** 그 노트북에서 `런타임 → 런타임 연결 해제 및 삭제`를 먼저 하세요. 무료 코랩은 GPU 세션을 하나만 허용합니다.

**셀 배지**: ⚙️ 준비 · 🔵 따라 실행 · 🟡 바꿔서 실행 · 🤔 생각해보기 · 🟠 심화(시간 남을 때)

## 📄 실습 원문 — 한빛테크 출장·복리후생 규정
이 노트북은 **같은 규정집 한 부**를 두 가지 형태로 읽습니다. 가상 회사 '한빛테크' 인사팀의 규정(교육용 가상 문서, 2026년 개정판)입니다.

| 파일 | 형태 | 내용 |
|---|---|---|
| `hanbit_rules.pdf` | 디지털 PDF, A4 6쪽 | 글자 정보가 들어 있는 PDF(워드에서 PDF로 저장한 파일과 같은 종류). 6개 장, 표 7개(표 0~6) |
| `scan/p1.png` ~ `p6.png` | 스캔본 그림 6장 | 위 PDF를 쪽마다 200dpi 그림(1654×2339px)으로 만들고 −1.5°~+1.5° 기울기·잡음·흐림을 넣었다. 글자 정보가 없다 |
| `tables/t0_history.csv` ~ `t6_education.csv` | 정답 표 7개 | 표마다 원래 값. 파서 결과를 이것과 칸 단위로 맞춰 본다 |
| `hanbit_rules_gt_nfd.md` | 본문 텍스트 | 규정집 본문을 자모 분리형(NFD)으로 저장한 파일. 5절에서 쓴다 |

**쪽별 구성과 이 노트북에서 쓰는 곳**

| 쪽 | 장 | 표 | 표 모양 | 쓰는 곳 |
|---|---|---|---|---|
| 1 | 제1장 총칙 | 표 0 개정 이력 | 괘선 | 8절에서 골라 실행 |
| 2 | 제2장 국내 출장 | **표 1 국내 출장비** (직급 6행 × 4열) | 괘선: 모든 칸에 선 | **1~4절** 파서 비교 · **6~7절** 스캔 OCR(`scan/p2.png`) |
| 3 | 제3장 해외 출장 | **표 2 해외 출장비** (지역 5행 × 4열) | **무괘선**: 머리글 밑줄과 마지막 줄만. 머리글은 흰 글자라 화면에 안 보임 | **1~4절** 파서 비교 |
| 4 | 제4장 휴가 | 표 3 연차휴가 · 표 4 경조사 휴가 | 괘선, 한 쪽에 표 2개 | **8절** 바꿔서 실행 기본값(표 4, `scan/p4.png`) |
| 5 | 제5장 복리후생 | 표 5 경조금 · 표 6 교육비 | 괘선 | 8절에서 골라 실행 |
| 6 | 제6장 신청과 정산 절차 | 없음 | — | — |

> 📌 **비교 범위** — 1~4절은 **표 1과 표 2 두 개만** 비교합니다. 파서의 성패는 표에 **선이 있느냐**에서 갈립니다. 표 7개 중 6개(표 0·1·3·4·5·6)는 괘선 표이고, 무괘선 표는 표 2 하나뿐입니다. 그래서 괘선 표의 대표로 표 1을, 무괘선 표로 표 2를 씁니다. 나머지 괘선 표(표 3·4 등)도 표 1과 같은 결과가 나오는지는 8절에서 직접 바꿔 확인합니다.

## 0. 준비 ⚙️
파서 3종(pdfplumber · PyMuPDF/pymupdf4llm · opendataloader-pdf)과 EasyOCR을 설치합니다. EasyOCR은 첫 실행 때 모델(검출 + 한국어 인식, 약 100MB)을 받습니다. 설치 중 빨간 `ERROR: pip's dependency resolver …` 문구가 보여도, 코랩에 미리 깔린 다른 패키지(gradio · google-colab 등)와 버전이 달라 나오는 안내라 실습에는 영향이 없습니다.

**채점 방법(셀 일치율)** — 정답 표의 행마다 파서 결과에서 가장 비슷한 행을 찾고, **같은 열 자리에 같은 값**이 나온 칸을 셉니다. 표 1은 머리글 포함 7행 × 4열 = **28칸**, 표 2는 6행 × 4열 = **24칸**입니다. 공백, 굵게 표시(`**`), 칸 안 줄바꿈 표시(`<br>`)는 지우고 비교합니다.

In [ ]:
%pip -q install pdfplumber pymupdf pymupdf4llm opendataloader-pdf easyocr

In [ ]:
# ⚙️ GPU 확인 — T4가 잡혔는지, 메모리가 얼마인지
import torch, subprocess
HAS_GPU = torch.cuda.is_available()
if HAS_GPU:
    name = torch.cuda.get_device_name(0)
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU: {name} · 메모리 {total_gb:.1f} GB")
else:
    print("GPU 없음 — CPU로 진행합니다(느립니다). 런타임 유형에서 T4를 선택하면 빨라집니다.")
DEVICE = "cuda" if HAS_GPU else "cpu"
DTYPE = torch.float16 if HAS_GPU else torch.float32   # T4는 bf16을 지원하지 않아 fp16을 씁니다

In [ ]:
# ⚙️ 실습 데이터 로더 — 깃허브(gunyang-ai-api/data)에서 자동으로 받아옵니다. 업로드 불필요.
import os, json, urllib.request
from pathlib import Path
DATA_BASE = os.environ.get("GENAI_BASE", "https://raw.githubusercontent.com/acho98/gunyang-ai-api/main/data/")
CACHE = Path("genai_data"); CACHE.mkdir(exist_ok=True)

def fetch(name: str) -> str:
    """파일 이름(예: 'hanbit_rules.pdf', 'scan/p2.png') → 로컬 경로. URL이면 내려받아 캐시."""
    if DATA_BASE.startswith("http"):
        dst = CACHE / name
        if not dst.exists():
            dst.parent.mkdir(parents=True, exist_ok=True)
            urllib.request.urlretrieve(DATA_BASE + name, dst)
        return str(dst)
    return str(Path(DATA_BASE) / name)

def load_json(name: str):
    with open(fetch(name), encoding="utf-8") as f:
        return json.load(f)

def load_text(name: str) -> str:
    with open(fetch(name), encoding="utf-8") as f:
        return f.read()

# 한글 폰트(차트용) — gunyang-data의 BMDOHYEON.ttf 를 자동으로 받아 적용. 실패해도 진행.
def set_korean_font():
    import matplotlib.pyplot as plt
    from matplotlib import font_manager as fm
    font_url = "https://raw.githubusercontent.com/acho98/gunyang-data/main/BMDOHYEON.ttf"
    dst = CACHE / "BMDOHYEON.ttf"
    try:
        if not dst.exists():
            local = Path(DATA_BASE).parent / "BMDOHYEON.ttf" if not DATA_BASE.startswith("http") else None
            if local and local.exists():
                dst.write_bytes(local.read_bytes())
            else:
                urllib.request.urlretrieve(font_url, dst)
        fm.fontManager.addfont(str(dst))
        plt.rcParams["font.family"] = fm.FontProperties(fname=str(dst)).get_name()
        plt.rcParams["axes.unicode_minus"] = False
    except Exception as e:
        print("(안내) 한글 폰트 설정 실패 — 영문 라벨은 정상:", e)
set_korean_font()
print("데이터 소스:", DATA_BASE)

In [ ]:
# ⚙️ 채점 도구 — 파서가 뽑은 표를 정답 CSV와 칸 단위로 비교합니다
import re, pandas as pd

def norm(x):
    """공백·굵게 표시(**)·칸 안 줄바꿈 표시(<br>)를 지우고 비교. '일비 (원/일)'과 '일비(원/일)'은 같은 칸으로 봅니다."""
    return re.sub(r"\s+|\*\*|<br\s*/?>", "", "" if x is None else str(x))

def gt_rows(name):
    """정답 CSV → [머리글, 1행, 2행, …] (문자열 리스트의 리스트)"""
    df = pd.read_csv(fetch(f"tables/{name}.csv"), dtype=str).fillna("")
    return [list(df.columns)] + df.values.tolist()

def match_rate(name, rows):
    """정답의 각 행마다 가장 비슷한 추출 행을 찾아, 같은 열 위치에 같은 값이 나온 칸의 비율(0~1)."""
    gt = [[norm(c) for c in r] for r in gt_rows(name)]
    rows = [[norm(c) for c in r] for r in (rows or [])]
    if not rows:
        return 0.0
    hit = sum(max(sum(1 for j, c in enumerate(g) if j < len(r) and r[j] == c) for r in rows) for g in gt)
    return hit / sum(len(g) for g in gt)

def md_tables(md):
    """마크다운 문자열 안의 | 표 | 들을 [[칸, …], …] 리스트로 꺼냅니다(구분선 |---| 은 제외). 빈 칸도 빈 문자열로 남깁니다."""
    tabs, cur = [], []
    for line in md.splitlines() + [""]:
        s = line.strip()
        if s.startswith("|") and s.endswith("|"):
            cells = [c.strip() for c in s[1:-1].split("|")]
            if not all(re.fullmatch(r":?-{3,}:?", c) for c in cells if c):
                cur.append(cells)
        elif cur:
            tabs.append(cur); cur = []
    return tabs

def table_after_caption(md, caption):
    """'표 1.' 같은 캡션 줄 다음부터 다음 캡션·제목 전까지에서 첫 마크다운 표를 찾습니다."""
    lines = md.splitlines()
    idx = [i for i, l in enumerate(lines) if caption in l]
    if not idx:
        return []
    seg = []
    for l in lines[idx[0] + 1:]:
        if re.search(r"표 \d+\.", l) or l.lstrip().startswith("#"):
            break
        seg.append(l)
    tabs = md_tables("\n".join(seg))
    return tabs[0] if tabs else []

PDF = fetch("hanbit_rules.pdf")

In [ ]:
# 🔵 원문 미리 보기 — 비교할 두 표를 디지털 PDF에서 잘라 보고, 정답 CSV와 나란히 놓습니다
import pymupdf
from PIL import Image

def table_png(page_no, caption, last_cell):
    """page_no(0부터 센 쪽)에서 캡션 줄부터 마지막 칸 글자까지 잘라 그림으로 만듭니다."""
    pg = pymupdf.open(PDF)[page_no]
    top = pg.search_for(caption)[0].y0 - 4
    bottom = pg.search_for(last_cell)[-1].y1 + 8
    pix = pg.get_pixmap(dpi=100, clip=pymupdf.Rect(45, top, pg.rect.width - 45, bottom))
    return Image.frombytes("RGB", (pix.width, pix.height), pix.samples)

for page_no, name, cap, last, kind in [(1, "t1_domestic", "표 1.", "200,000", "괘선"),
                                       (2, "t2_overseas", "표 2.", "기타 지역", "무괘선")]:
    rows = gt_rows(name)
    print(f"\n■ 원문 {page_no + 1}쪽 {cap[:-1]} ({kind}) — 정답 tables/{name}.csv: {len(rows)}행 × {len(rows[0])}열 = {len(rows) * len(rows[0])}칸(머리글 포함)")
    display(table_png(page_no, cap, last))
    display(pd.DataFrame(rows[1:], columns=rows[0]))

💡 **해석** — 표 1은 모든 칸이 선으로 둘러싸여 있습니다. 표 2는 다른 점이 두 가지입니다.
- **선이 없다**: 머리글 밑줄과 마지막 줄 아래 선만 있고, 칸 사이는 **여백으로만** 나뉩니다.
- **머리글이 안 보인다**: 그림에서 표 2 맨 위 줄이 비어 있지요? 머리글(`지역 · 일비 (과장 이하) · …`)이 **흰 글자에 배경 없이** 찍혀 있어 화면에는 보이지 않고, 글자는 PDF 속 텍스트 레이어에만 있습니다(이론2 p15). 정답 CSV에는 머리글이 있습니다.

사람 눈에는 둘 다 표이지만, 파서는 이 차이에서 갈립니다. 아래 1~4절의 셀 일치율은 모두 이 두 정답 표와 비교한 값입니다.

## 1. pdfplumber — 선이 있는 표는 잘 잡는다 🔵
`pdfplumber`(MIT)는 PDF 안의 글자 좌표와 **선(line)** 을 읽어 표를 찾습니다. 원문 **2쪽 표 1(괘선)** 과 **3쪽 표 2(무괘선)** 를 차례로 넣습니다. 코드의 쪽 번호는 0부터 셉니다(`pages[1]` = 2쪽, `pages[2]` = 3쪽).

In [ ]:
import pdfplumber
results = {}   # (파서, 표) → 셀 일치율

with pdfplumber.open(PDF) as pdf:
    print(f"총 {len(pdf.pages)}쪽\n--- 2쪽 본문 앞부분 ---")
    print(pdf.pages[1].extract_text()[:260], "…")
    for page_no, name in [(1, "t1_domestic"), (2, "t2_overseas")]:
        tables = pdf.pages[page_no].extract_tables()
        print(f"\n{page_no+1}쪽: 찾은 표 {len(tables)}개")
        if tables:
            display(pd.DataFrame(tables[0][1:], columns=tables[0][0]))
        results[("pdfplumber", name)] = max([match_rate(name, t) for t in tables], default=0.0)
print({k[1]: f"{v:.0%}" for k, v in results.items()})

💡 **해석** — 2쪽 표 1은 28칸 모두 정답과 같았고(**100%**), 3쪽 표 2는 **표를 0개** 찾았습니다(0%). 본문 줄바꿈도 PDF에 찍힌 위치 그대로 들어옵니다. pdfplumber는 "선으로 둘러싼 칸"을 표로 보기 때문에, 선을 빼고 여백만으로 정렬한 표 2는 표로 인식하지 못합니다. **디지털 PDF + 괘선 표**라면 무료 파서로 충분하다는 것이 첫 번째 교훈입니다(이론2 p14).

## 2. PyMuPDF · pymupdf4llm — 표 찾기와 마크다운 변환 🔵
PyMuPDF는 빠른 PDF 엔진이고, `find_tables()`로 표를 DataFrame으로 꺼냅니다. `pymupdf4llm`은 PDF를 **LLM용 마크다운**으로 한 번에 바꿉니다. 대상은 1절과 같은 2쪽 표 1 · 3쪽 표 2입니다.
> ⚖️ PyMuPDF·pymupdf4llm은 **AGPL** 라이선스입니다. 사내 서비스에 넣으려면 소스 공개 의무나 상용 라이선스를 검토해야 합니다.

In [ ]:
import pymupdf
doc = pymupdf.open(PDF)
for page_no, name in [(1, "t1_domestic"), (2, "t2_overseas")]:
    found = doc[page_no].find_tables().tables
    print(f"{page_no+1}쪽: 찾은 표 {len(found)}개")
    dfs = [t.to_pandas() for t in found]
    if dfs:
        display(dfs[0])
    results[("PyMuPDF find_tables", name)] = max([match_rate(name, [list(d.columns)] + d.values.tolist()) for d in dfs], default=0.0)

In [ ]:
import pymupdf4llm

def to_md(pages):
    """pymupdf4llm으로 마크다운 변환(pages는 0부터 센 쪽 번호 리스트).
    pymupdf4llm은 불러오는 순간 PyMuPDF 전역 설정 두 가지(레이아웃 모드 켬, quad 보정 끔)를 바꿉니다.
    그대로 두면 위 find_tables() 결과까지 달라집니다(코랩 실측: 표 1 100% → 36%). 변환할 때만 켜고 끝나면 되돌립니다."""
    pymupdf4llm.use_layout(True)
    try:
        return pymupdf4llm.to_markdown(PDF, pages=pages)
    finally:
        pymupdf4llm.use_layout(False)
        pymupdf.TOOLS.unset_quad_corrections(False)

md_4llm = to_md([1, 2])                                    # 2·3쪽만 마크다운으로
print(md_4llm[-900:])                                      # 뒷부분(표 2 근처)만 출력
for name, cap in [("t1_domestic", "표 1."), ("t2_overseas", "표 2.")]:
    results[("pymupdf4llm", name)] = match_rate(name, table_after_caption(md_4llm, cap))

💡 **해석** — `find_tables()`도 표 1은 완벽하지만 표 2는 0개입니다(같은 '선 기반' 방식). 반면 `pymupdf4llm`은 레이아웃 모델(pymupdf-layout)을 함께 써서 **표 2의 본문 5행을 표로 살려 냅니다.** 단, 출력을 보면 머리글 행(`지역 | 일비 (과장 이하) | …`)이 없습니다. 화면에 보이지 않는 흰 머리글을 표에 넣지 않은 것입니다. 24칸 중 머리글 4칸이 빠져 20칸이 맞았습니다(83%). 숫자는 있는데 **어느 열이 무엇인지 모르는 표**가 된 것이죠. RAG에서 "차장 이상 북미 일비?"를 물으면 틀릴 수 있는 상태입니다.
- 참고: 같은 PDF에서 머리글 색만 보이는 색(남색)으로 바꿔 돌려 보면 pymupdf4llm은 24칸을 모두 맞힙니다(2026-10-18 강사 실험). 머리글이 빠진 원인은 선이 없어서가 아니라 **머리글이 보이지 않아서**입니다.

> ⚠️ 위 `to_md()`가 설정을 되돌리는 이유: 라이브러리 하나를 불러온 것만으로 **다른 라이브러리의 결과가 바뀔 수 있습니다.** 파서를 여러 개 비교할 때는 각 파서를 따로 돌려 본 값과 같은지 확인하세요.

## 3. opendataloader-pdf — 읽기 순서 + JSON 🔵
`opendataloader-pdf`(Apache 2.0)는 XY-Cut++ 읽기 순서와 요소별 JSON(제목·문단·표·좌표)을 함께 줍니다. 규정집 6쪽 전체를 한 번에 변환하고, 그중 표 1 · 표 2만 채점합니다. **Java 11 이상**이 필요합니다. 코랩에 Java가 없으면 설치를 시도하고(약 1분), 실패하면 이 셀은 건너뜁니다.

In [ ]:
import shutil, subprocess, glob
HAS_JAVA = shutil.which("java") is not None
if not HAS_JAVA and os.path.exists("/content"):          # 코랩에서만 자동 설치
    print("Java가 없어 설치합니다(약 1분)…")
    subprocess.run("apt-get -qq update && apt-get -qq install -y openjdk-17-jre-headless",
                   shell=True, capture_output=True, text=True)
    HAS_JAVA = shutil.which("java") is not None

md_odl = ""
if HAS_JAVA:
    try:
        import opendataloader_pdf
        opendataloader_pdf.convert(input_path=[PDF], output_dir="odl_out", format="markdown,json")  # 로그가 많이 찍혀도 정상
        md_odl = open(glob.glob("odl_out/*.md")[0], encoding="utf-8").read()
        odl_json = json.load(open(glob.glob("odl_out/*.json")[0], encoding="utf-8"))
        print("JSON 요소 종류:", sorted({k["type"] for k in odl_json["kids"]}), "· 요소 수", len(odl_json["kids"]))
        i = md_odl.find("표 2.")
        print("--- 표 2 주변 마크다운 ---\n" + md_odl[i - 2:i + 260])
    except Exception as e:
        print("opendataloader 실행 실패 — 건너뜀:", type(e).__name__, str(e)[:200])
else:
    print("Java 없음 — opendataloader 셀은 건너뜀")
if md_odl:
    for name, cap in [("t1_domestic", "표 1."), ("t2_overseas", "표 2.")]:
        results[("opendataloader", name)] = match_rate(name, table_after_caption(md_odl, cap))

💡 **해석** — 표 1은 역시 100%입니다. 표 2는 `- 표 2. 해외 출장비 … 지역 일비 (과장 이하) … 북미 60 80 220 유럽 …`처럼 **한 줄 문장으로 펴져** 버렸습니다(일치율 0%). 글자는 하나도 빠지지 않았지만(화면에 안 보이는 머리글 글자까지 들어 있음) 행·열 구조가 사라졌습니다. 그래도 JSON에 제목·목록·표가 요소별로 나뉘어 있어 **청킹 경계**를 잡기에는 좋습니다.

## 4. 한눈에 비교 — 표 1(괘선) vs 표 2(무괘선) 🔵

In [ ]:
cmp = (pd.Series(results).unstack().rename(columns={"t1_domestic": "표1 국내(괘선)", "t2_overseas": "표2 해외(무괘선)"}))
display((cmp * 100).round(0).astype(int).astype(str) + "%")

💡 **해석** — 강사 환경과 코랩 T4의 결과가 같습니다.

| 파서 | 표1 (괘선, 28칸) | 표2 (무괘선, 24칸) | 표2에서 무슨 일이? |
|---|---|---|---|
| pdfplumber | 100% | 0% | 표를 못 찾음 |
| PyMuPDF `find_tables` | 100% | 0% | 표를 못 찾음 |
| pymupdf4llm | 100% | 83% | 본문 5행은 표로 살리고, **보이지 않는 머리글 행은 뺌** |
| opendataloader | 100% | 0% | 한 줄 문장으로 펴짐 |

- **괘선 표는 네 파서 모두 100%**입니다. 이 규정집의 다른 괘선 표(표 0·3·4·5·6)도 네 파서 모두 100%였습니다(코랩 실측, 8절에서 직접 확인).
- 차이는 표 2처럼 **선이 없고 머리글이 화면에 안 보이는** 표에서 납니다.
- pdfplumber·PyMuPDF의 0%는 **기본 설정(선 기준)** 의 결과입니다. 표 영역을 지정하고 글자 정렬 기준(`text` 전략)으로 바꾸면 96%까지 올라갑니다(이론2 p16·p18). 표 0개는 도구의 한계가 아니라 설정의 결과입니다.
- 개발사가 공개한 벤치마크(이론2 p12)보다 **내 문서로 직접 재 보는 것**이 정답입니다.

## 5. 유니코드 정규화(NFC) — 눈에 같은데 검색이 안 된다 🔵
macOS에서 만든 파일이나 일부 시스템 export는 한글을 **자모 분리형(NFD)** 으로 저장합니다. 규정집 본문을 NFD로 저장한 파일(`hanbit_rules_gt_nfd.md`)에서 "출장"을 검색해 봅니다.

In [ ]:
import unicodedata
nfd_text = load_text("hanbit_rules_gt_nfd.md")
print("화면에는 똑같이 보임:", nfd_text[:30])
print('"출장" in 원본(NFD)      →', "출장" in nfd_text, f"(글자 수 {len(nfd_text):,})")
fixed = unicodedata.normalize("NFC", nfd_text)
print('"출장" in NFC 정규화 후  →', "출장" in fixed, f"(글자 수 {len(fixed):,}) · 등장 {fixed.count('출장')}회")

💡 **해석** — NFD는 '출'을 `ㅊ+ㅜ+ㄹ` 세 코드로 저장합니다. 그래서 키보드로 친 "출장"(NFC)과 **바이트가 달라** `in` 검색, BM25, 임베딩 모두 엉뚱하게 동작합니다. 파서 출력이든 OCR 출력이든 **저장 전에 `unicodedata.normalize("NFC", s)` 한 줄**을 넣으세요(노트북 02 4절, 이론1 p127).

## 6. 스캔본은 OCR — EasyOCR 🔵
입력은 **`scan/p2.png`** 입니다. 디지털 PDF 2쪽(제2장 본문 + 표 1)을 스캔한 것처럼 만든 그림입니다(1654×2339px, 약 1° 기울어짐, 잡음). 글자 정보가 없는 **그림**이라 PDF 파서로는 아무것도 못 꺼냅니다. EasyOCR은 **글자 영역 검출(CRAFT) → 글자 인식** 두 단계로 `(박스 4점, 글자, 신뢰도)` 목록을 돌려줍니다(이론2 p19).
- **박스 4점**: 글자 영역 네 꼭짓점의 픽셀 좌표(왼쪽 위부터 시계 방향). 7절에서 이 좌표로 표를 다시 짭니다
- **글자**: 인식한 문자열
- **신뢰도(conf)**: 모델이 낸 0~1 점수. 6-3에서 무엇에 쓰는지 봅니다

### 6-1. 글자·좌표·신뢰도 읽기

In [ ]:
import easyocr, time
t0 = time.time()
reader = easyocr.Reader(["ko", "en"], gpu=HAS_GPU)        # 첫 실행 때 모델 다운로드(약 100MB)
SCAN = fetch("scan/p2.png")
ocr = reader.readtext(SCAN)                               # [(bbox 4점, text, conf), …]
print(f"검출 {len(ocr)}개 · {time.time()-t0:.1f}초 ({DEVICE})\n")
for bbox, text, conf in ocr[:12]:
    x, y = map(int, bbox[0])
    print(f"({x:4d},{y:4d})  conf={conf:.2f}  {text}")

💡 **해석** — 강사 환경(Mac CPU)에서 2쪽 한 장에 **40개 박스**, 시간은 10초~1분대(다른 작업 부하에 따라)였습니다. 코랩 T4에서는 약 10초였습니다(실측 9.8~10.6초, 첫 실행의 모델 다운로드 포함). 본문 줄을 보면 `구성한다 → 구성하다`, `택시를 → 택시틀`, `을 → 올`처럼 **받침 하나가 틀린 오류**가 많고 신뢰도도 0.3~0.5로 낮습니다. 반면 `사원`, `20,000` 같은 짧은 표 칸은 신뢰도 0.7~1.0으로 높습니다(6-2 그림의 초록 박스). 기울기와 잡음이 섞인 스캔본의 현실입니다.

### 6-2. 박스를 그림 위에 그리기 — 색은 신뢰도
초록 = 0.7 이상 · 주황 = 0.5~0.7 · 빨강 = 0.5 미만. 위쪽 절반(본문 + 표 1)만 보여 줍니다.

In [ ]:
from PIL import Image, ImageDraw
img = Image.open(SCAN).convert("RGB")
draw = ImageDraw.Draw(img)
for bbox, text, conf in ocr:
    color = (0, 160, 0) if conf >= 0.7 else (230, 140, 0) if conf >= 0.5 else (220, 0, 0)   # 초록=높음, 주황, 빨강=낮음
    draw.polygon([tuple(map(int, p)) for p in bbox], outline=color, width=4)
display(img.crop((0, 0, img.width, 1150)).resize((img.width // 2, 575)))   # 위쪽 절반만 축소해서 표시

💡 **해석** — 본문 줄은 대부분 빨강, 표의 숫자 칸은 대부분 초록입니다. 표 머리글 중 `일비 (원/일)` · `식비 (원/일)` · `숙박비 상한 (원/박)` 3칸도 빨강입니다.

### 6-3. 신뢰도(conf)는 어디에 쓰나
OCR이 틀린 곳을 찾으려고 박스 40개를 사람이 전부 볼 수는 없습니다. 그래서 두 가지를 확인합니다.
- **질문 1 (찾기)**: 신뢰도가 낮은 박스만 보면 틀린 곳을 다 찾을 수 있을까?
- **질문 2 (버리기)**: 신뢰도가 낮은 박스를 그냥 버려도 될까?

**맞음·틀림 판정** — 같은 쪽의 디지털 PDF(2쪽) 글자가 정답입니다. 공백을 지운 OCR 글자가 정답 글자 안에 **그대로** 있으면 '맞음', 글자나 기호가 하나라도 다르면 '틀림'입니다.

In [ ]:
# 질문 1 — 박스마다 맞음·틀림을 붙이고, 신뢰도 구간별로 셉니다
with pdfplumber.open(PDF) as pdf:
    PAGE_TEXT = [p.extract_text() for p in pdf.pages]      # 디지털 PDF 쪽별 글자 = 스캔본의 정답

def check_ocr(ocr, page):
    """page는 1부터 센 쪽 번호. 공백을 지운 OCR 글자가 그 쪽 원문 글자 안에 그대로 있으면 맞음."""
    ref = re.sub(r"\s+", "", PAGE_TEXT[page - 1])
    df = pd.DataFrame([{"conf": c, "맞음": re.sub(r"\s+", "", t) in ref, "글자": t[:40]} for _, t, c in ocr])
    df["신뢰도 구간"] = pd.cut(df["conf"], [0, 0.5, 0.7, 1.01], right=False, labels=["0.5 미만", "0.5~0.7", "0.7 이상"])
    return df

chk = check_ocr(ocr, page=2)
print(f"박스 {len(chk)}개 중 틀림 {int((~chk['맞음']).sum())}개")
by_band = chk.groupby("신뢰도 구간", observed=False)["맞음"].agg(박스="size", 맞음="sum")
display(by_band.assign(틀림=by_band["박스"] - by_band["맞음"]))
print("틀리게 읽은 박스 (신뢰도 높은 순):")
display(chk[~chk["맞음"]].sort_values("conf", ascending=False).round(2))

In [ ]:
# 질문 2 — 신뢰도가 임계값보다 낮은 박스를 버리면 무엇이 남고 무엇이 사라질까?
rows = []
for th in [0.3, 0.5, 0.7]:
    drop, keep = chk[chk["conf"] < th], chk[chk["conf"] >= th]
    rows.append({"임계값": th, "버린 박스": len(drop), "버린 것 중 맞던 박스": int(drop["맞음"].sum()),
                 "남은 틀린 박스": int((~keep["맞음"]).sum())})
display(pd.DataFrame(rows))
print("conf < 0.5 로 버려지는 박스:")
for _, r in chk[chk["conf"] < 0.5].iterrows():
    print(f"  {r['conf']:.2f}  {'맞음' if r['맞음'] else '틀림'}  {r['글자']}")

💡 **해석** — 코랩 T4 실측, 2쪽 박스 40개 중 틀린 것은 11개입니다.
- **질문 1 (찾기)**: conf 0.7 이상 27개는 모두 맞았고, 틀린 11개는 전부 0.7 미만 13개 안에 있었습니다. 40개를 다 보지 않고 **13개만 검수하면** 이 쪽의 오류를 모두 잡습니다. 신뢰도는 이렇게 쓰는 값입니다.
- 단, 낮다고 다 틀린 것은 아닙니다. `110,000`은 맞게 읽고도 0.41이었습니다(이론2 p20).
- **질문 2 (버리기)**: 0.5 미만을 버리면 틀린 박스 9개가 사라집니다. 그러나 고쳐진 것이 아니라 **없어진** 것입니다. 버려지는 10개는 제5조~제7조 본문 6줄, 표 머리글 3칸, 그리고 맞게 읽은 `110,000` 1개입니다. 이 결과를 RAG에 넣으면 "자가 차량은 1km당 얼마?"(제6조)에 답할 근거가 아예 없습니다.
- **정리**: 신뢰도는 **버릴 기준이 아니라, 사람이 검수하거나 더 강한 도구(VLM 등)로 다시 읽을 곳을 고르는 기준**입니다. 신뢰도가 높은데도 틀리는 경우는 8절(4쪽)에서 봅니다.

### 6-4. 박스를 문단으로 합치면 — 구조는 남을까

In [ ]:
para = reader.readtext(SCAN, detail=0, paragraph=True)   # 가까운 박스를 문단으로 합치고 글자만
print(f"문단 {len(para)}개 · 문단별 글자 수 {[len(p) for p in para]}")
big = max(para, key=len)
i = big.find("직급")                                       # 표 1 머리글이 시작하는 곳
print("--- 가장 긴 문단에서 표 1이 시작하는 부분 ---")
print("…", big[max(i - 120, 0):i + 260], "…")

💡 **해석** — `paragraph=True`는 가까운 박스를 문단으로 합쳐 줍니다. 그런데 결과를 보면 제7조 본문과 **표 1이 한 문단으로 뭉쳐** `… 교통비만 지급하다. 표 1. 국내 출장비 지급 기준 (단위: 원) 직급 일비 (원 일) … 사원 20,000 30,000 80,000 대리 …`처럼 숫자가 줄줄이 이어집니다. 어느 숫자가 어느 직급의 어느 항목인지 알 수 없습니다. OCR은 **글자는 주지만 구조(문단·표·읽기 순서)는 주지 않습니다**(이론2 p22). 그리고 `원/일`의 `/`가 사라진 것도 보이나요?

## 7. OCR 결과로 표 1을 다시 표로 만들 수 있을까 🔵
6절에서 본 것처럼 OCR은 **글자와 좌표만** 주고, "몇 행 몇 열"은 알려 주지 않습니다(이론2 p22). 여기서는 가장 단순한 규칙으로 표 1을 직접 다시 짜 보고, 이 방법이 왜 실무에서 안 쓰이는지 확인합니다.

- **입력**: 스캔 2쪽 OCR 박스 중 "표 1." 캡션 아래에 있는 박스 28개 (머리글 4칸 + 직급 6행 × 4칸)
- **규칙**: 박스 왼쪽 위 꼭짓점의 y좌표가 그 행 첫 박스와 **`tol` 픽셀 안**이면 같은 행으로 묶는다. 행 안에서는 x좌표 순서로 칸을 놓는다
- **정답**: 머리글 포함 **7행 × 4열**. 묶인 행 수가 7이 되는지, 셀 일치율이 몇 %인지 본다
- **바꿔 보는 값**: `tol` = 15 · 30 · 60px

In [ ]:
cap_y = next(b[0][1] for b, t, c in ocr if t.replace(" ", "").startswith("표1"))
cells = [(b, t, c) for b, t, c in ocr if b[0][1] > cap_y + 20]          # 캡션 아래 = 표 영역
print(f"표 영역 박스 {len(cells)}개 (정답 표는 7행 × 4열 = 28칸)")
display(img.crop((100, int(cap_y) - 10, 1550, 1090)).resize((725, 240)))

def rows_by_y(items, tol):
    """y가 행의 첫 박스와 tol 픽셀 이내면 같은 행. 행 안에서는 x 순서로 정렬."""
    rows, start = [], None
    for b, t, c in sorted(items, key=lambda x: x[0][0][1]):
        y = b[0][1]
        if start is None or y - start > tol:
            rows.append([]); start = y
        rows[-1].append((b[0][0], t))
    return [[t for x, t in sorted(r)] for r in rows]

ys = {t: int(b[0][1]) for b, t, c in cells}                               # 글자 → 박스 왼쪽 위 y
print(f"기울기: 과장 행 '과장' y={ys.get('과장')} · 맨 오른쪽 '110,000' y={ys.get('110,000')} | 행 간격: '사원' y={ys.get('사원')} → '대리' y={ys.get('대리')}")
for tol in [15, 30, 60]:
    rows = rows_by_y(cells, tol)
    print(f"tol={tol:>2}px → {len(rows):>2}행 (정답 7행) · 셀 일치율 {match_rate('t1_domestic', rows):.0%}")
print("\n[tol=15] 앞 4행 — 한 행이 둘로 쪼개진 모습")
display(pd.DataFrame(rows_by_y(cells, 15)).head(4))
print("[tol=30] 전체 — 행 수가 정답과 같음")
display(pd.DataFrame(rows_by_y(cells, 30)))

💡 **해석** — 강사 환경과 코랩 T4의 결과가 같습니다.

| tol | 묶인 행 | 무슨 일이 | 셀 일치율 |
|---|---|---|---|
| 15px | 14행 | 행마다 둘로 쪼개짐. 숙박비 칸이 다음 행으로 떨어진다(위 표에서 `80,000`이 혼자 한 행) | 68% |
| 30px | **7행** | 정답과 같은 행 수 | **89%** |
| 60px | 5행 | 위아래 행이 한 행으로 합쳐짐 | 39% |

- **쪼개지는 이유**: 스캔 2쪽은 약 1° 기울어 있어, 같은 행이라도 오른쪽 칸일수록 y가 커집니다(위 '기울기' 줄). 과장 행에서 `과장`은 y=852, 맨 오른쪽 `110,000`은 y=870으로 18px 차이입니다. tol=15면 숙박비 칸이 새 행이 됩니다.
- **합쳐지는 이유**: 행 간격이 약 56px(사원 y=740 → 대리 y=796)이라, tol=60이면 아래 행 박스까지 같은 행으로 묶입니다.
- **30px에서도 100%가 아닌 이유**: 머리글 3칸에서 OCR이 `원/일`의 `/`를 못 읽어(`원 일`) 28칸 중 25칸만 맞았습니다(89%).
- **결론**: 알맞은 `tol`은 기울기와 행 간격으로 정해지므로 쪽마다·문서마다 다릅니다. 이런 규칙은 다른 쪽에서 바로 깨집니다. 그래서 실무에서는 **레이아웃 모델로 표 영역을 찾고, 표 구조 모델(TSR)로 행·열을 나눕니다**(이론2 p23, 노트북 07·08).

## 8. 바꿔서 실행 🟡
1. **다른 표로 재비교** — `PAGE_NO, TABLE, CAPTION`을 아래 표에서 골라 바꾸세요. 기본값은 4쪽 표 4입니다. 4쪽에는 표 3·4가 함께 있어서, 파서가 한 쪽의 표 2개를 따로 찾는지도 볼 수 있습니다.

| 원문 쪽 | `PAGE_NO` | 표 | `TABLE` | `CAPTION` |
|---|---|---|---|---|
| 1쪽 | `0` | 표 0 개정 이력 | `"t0_history"` | `"표 0."` |
| 2쪽 | `1` | 표 1 국내 출장비 | `"t1_domestic"` | `"표 1."` |
| 3쪽 | `2` | 표 2 해외 출장비 (무괘선, 머리글 안 보임) | `"t2_overseas"` | `"표 2."` |
| 4쪽 | `3` | 표 3 연차휴가 | `"t3_annual_leave"` | `"표 3."` |
| 4쪽 | `3` | 표 4 경조사 휴가 | `"t4_family_leave"` | `"표 4."` |
| 5쪽 | `4` | 표 5 경조금 | `"t5_condolence"` | `"표 5."` |
| 5쪽 | `4` | 표 6 교육비 | `"t6_education"` | `"표 6."` |

In [ ]:
PAGE_NO, TABLE, CAPTION = 3, "t4_family_leave", "표 4."     # ← 바꿔 보세요 (PAGE_NO는 0부터 센 쪽 번호)
r2 = {}
with pdfplumber.open(PDF) as pdf:
    tabs = pdf.pages[PAGE_NO].extract_tables()
r2["pdfplumber"] = max([match_rate(TABLE, t) for t in tabs], default=0.0)
dfs = [t.to_pandas() for t in pymupdf.open(PDF)[PAGE_NO].find_tables().tables]
r2["PyMuPDF find_tables"] = max([match_rate(TABLE, [list(d.columns)] + d.values.tolist()) for d in dfs], default=0.0)
r2["pymupdf4llm"] = match_rate(TABLE, table_after_caption(to_md([PAGE_NO]), CAPTION))
if md_odl:
    r2["opendataloader"] = match_rate(TABLE, table_after_caption(md_odl, CAPTION))
rows = gt_rows(TABLE)
print(f"원문 {PAGE_NO+1}쪽 {CAPTION[:-1]} (정답 {len(rows)}행 × {len(rows[0])}열 = {len(rows) * len(rows[0])}칸) · 이 쪽에서 찾은 표 수 — pdfplumber {len(tabs)}개 · PyMuPDF {len(dfs)}개")
display(pd.Series(r2, name=TABLE).map("{:.0%}".format).to_frame())

💡 **해석** — 기본값(4쪽 표 4, 8행 × 3열 = 24칸)은 네 파서 모두 **100%** 였고, pdfplumber와 PyMuPDF 모두 4쪽에서 표 2개(표 3·4)를 따로 찾았습니다(코랩 실측). 괘선 표(표 0·1·3·4·5·6)는 어느 것을 골라도 네 파서 모두 100%입니다. 결과가 갈리는 표는 무괘선인 표 2 하나뿐입니다.

2. **다른 스캔본 + 임계값** — `SCAN2`를 `scan/p4.png`(4쪽: 표 3·4)로 두고 `CONF_TH`를 0.3 · 0.5 · 0.8로 바꿔 보세요. 6-3과 같은 방법(같은 쪽 디지털 PDF 글자와 대조)으로 맞음·틀림을 셉니다.

In [ ]:
SCAN2, CONF_TH = "scan/p4.png", 0.5                     # ← 바꿔 보세요 (scan/p1.png ~ p6.png)
page2 = int(re.search(r"p(\d+)\.png", SCAN2).group(1))
chk2 = check_ocr(reader.readtext(fetch(SCAN2)), page=page2)
keep = chk2[chk2["conf"] >= CONF_TH]
print(f"{SCAN2}: 박스 {len(chk2)}개 · 틀림 {int((~chk2['맞음']).sum())}개")
print(f"conf ≥ {CONF_TH}: 남은 박스 {len(keep)}개 · 그중 틀린 박스 {int((~keep['맞음']).sum())}개 (신뢰도만 보면 검수에서 빠지는 오류)")
display(keep[~keep["맞음"]].sort_values("conf", ascending=False).round(2))

💡 **해석** — 기본값(4쪽, conf ≥ 0.5) 코랩 실측입니다. 박스 49개 중 44개가 남고, **남은 박스 안에도 틀린 것이 6개** 있습니다.
- 2쪽과 달리 4쪽은 conf 0.7 이상 40개 중에도 5개가 틀렸습니다.
- `76일`(정답 `16일`, conf 0.78), `7년 이상 3년 미만`(정답 `1년 이상 3년 미만`, 0.73)처럼 **숫자 1이 7로 바뀐 오류가 높은 신뢰도로** 나옵니다. 신뢰도만 믿으면 이 오류는 검수에서 빠집니다.
- `조부모 외조부모 사망`(0.97) · `본인 배우자의 부모 사망`(0.84)은 가운뎃점(·)이 빠져 '틀림'으로 셌습니다. 의미는 같지만 검색·대조에서는 다른 글자입니다. `축진하다.`(0.75, 정답 `촉진한다.`)도 높은 신뢰도의 오자입니다.
- 그래서 숫자는 신뢰도와 따로 **형식·업무 규칙**으로 검사합니다. 예: 근속이 길수록 연차 일수가 늘어야 한다(15 → **76** → 17일은 규칙 위반)(이론2 p24).

### 🤔 생각해보기

이 규정집이라면 **어느 층에서 멈춰도 될까요?** 디지털 PDF일 때와 스캔본일 때를 나눠 답해 보세요. (층: ① 기본 파서 → ② OCR → ③ 레이아웃·표 구조 → ④ VLM)

<details><summary><b>🤔 생각해보기 — 해설 보기(클릭)</b></summary>

- **디지털 PDF**: 표 7개 중 6개가 괘선 표이므로 ① 기본 파서로 충분합니다(비용 0, 수 초). 단, 무괘선 표 2처럼 깨지는 표가 **하나라도** 있으면 그 쪽만 ③(Docling 등)으로 보내거나 사람이 검수합니다. "전체를 무거운 도구로"가 아니라 "깨지는 곳만 위 층으로"가 비용 효율적입니다.
- **스캔본**: ②만으로는 표 구조가 사라지고(행·열 섞임) 받침·숫자 오류가 많습니다. 최소 ③(레이아웃 + 표 구조)이 필요하고, 한국어 품질이 중요하면 ④ 한국어를 지원하는 VLM(PaddleOCR-VL)이나 상용 OCR을 검토합니다. 신뢰도가 낮은 곳은 사람이 검수하고, 숫자는 업무 규칙으로 따로 검사합니다.
- 어느 경우든 **정답 몇 개를 만들어 셀 일치율로 재 보는 것**이 선택 근거가 됩니다(이론2 p47·p55).

</details>

---
### ✅ 이 노트북에서 가져갈 것
- 이 규정집의 괘선 표 6개는 pdfplumber·PyMuPDF·pymupdf4llm·opendataloader 모두 100%. 차이는 무괘선 표 2에서 난다.
- 파서는 글자를 꺼내도 구조(머리글·행·열)를 잃을 수 있다. 정답 CSV로 셀 일치율을 재라.
- 추출 텍스트는 저장 전에 NFC 정규화. NFD면 검색이 실패한다.
- OCR은 (박스, 글자, 신뢰도)만 준다. 신뢰도는 버릴 기준이 아니라 검수할 곳을 고르는 기준이고, 높은 신뢰도의 숫자도 틀릴 수 있다.
- 읽기 순서·표 구조는 다음 층(레이아웃·표 구조 모델·VLM)의 몫.

> 다음 노트북: **07_layout_vlm_parsing — 레이아웃 모델로 구조를 보고, VLM이 통째로 읽는다**